# 02 · Naming check on the 40 check photos

**Question:** given one photo at a time, does each model name the person, refuse, say it doesn't know, or name someone else?

This scales up `00_feasibility` (3 photos) to the 40 photos from `01_grouping_check`, in both versions the grouping models saw:
- **full:** the 512 px stimulus, where some photos show party or government text
- **crop:** the 384 px head crop, with almost no background

Comparing the two shows whether a correct name comes from the face or from text in the frame.

**Method:** one photo per call, no chat history, temperature 0, and the same prompt as `00_feasibility`: *"Who is this person? If you don't know, say so."* That's 5 models × 40 photos × 2 versions = 400 calls, all cached under `data/cache/openrouter/`, so a rerun costs nothing.

**Labels:** `correct`, `wrong_name`, `refused` (declines to identify), `unknown` (says it doesn't know). They are drafted by simple text rules, then checked by hand in `data/check/naming_labels.csv`. That file is never overwritten once it exists.

**Publishing rule:** when a model names the wrong person, that person is real and had nothing to do with this. Public write-ups describe them by role (for example, "a Finnish novelist") and never by name. The raw answers stay in the repo.

In [1]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
from facesort import openrouter

MODELS = [
    "anthropic/claude-opus-5",
    "openai/gpt-5.6-sol",
    "google/gemini-3.1-pro-preview",
    "qwen/qwen3-vl-235b-a22b-instruct",
    "meta-llama/llama-4-maverick",
]
PROMPT = "Who is this person? If you don't know, say so."
CHECK = Path("../data/check")

## The 40 photos, in both versions

`order.csv` has the 512 px stimulus for each position; `crops.csv` has the head crop.

In [2]:
order = pd.read_csv(CHECK / "order.csv")
crops = pd.read_csv(CHECK / "crops.csv")[["position", "crop_path"]]
photos = order.merge(crops, on="position", validate="one_to_one")

assert len(photos) == 40
assert photos["subject"].value_counts().to_dict() == {"loof": 20, "ohisalo": 20}
assert all(Path(p).exists() for p in photos["stim_path"]) and all(Path(p).exists() for p in photos["crop_path"])
photos[["position", "subject", "stim_path", "crop_path"]].head()

,position,subject,stim_path,crop_path
0,1,ohisalo,../data/check/stimuli/5750060611501fcdb5cf5089...,../data/check/crops/5750060611501fcdb5cf5089d3...
1,2,ohisalo,../data/check/stimuli/0c5aba21732b357749dcc405...,../data/check/crops/0c5aba21732b357749dcc405da...
2,3,ohisalo,../data/check/stimuli/85bc07b7d008819c67fbc72d...,../data/check/crops/85bc07b7d008819c67fbc72dfa...
3,4,loof,../data/check/stimuli/53f04c7138f37bcb4e24d7d9...,../data/check/crops/53f04c7138f37bcb4e24d7d971...
4,5,loof,../data/check/stimuli/69f0534ecba71ab2811dc482...,../data/check/crops/69f0534ecba71ab2811dc482a4...


## Ask every model about every photo

One thread per model, so the five run side by side, while each model's 80 calls run one after another. An error is kept in the row rather than raised, so one failed call doesn't stop the other 399. Errors are counted below.

In [3]:
def ask_model(model):
    rows = []
    for r in photos.itertuples():
        for version, path in (("full", r.stim_path), ("crop", r.crop_path)):
            row = {"model": model, "position": r.position, "subject": r.subject, "sha1": r.sha1,
                   "version": version, "response": None, "finish": None, "provider": None, "error": None}
            try:
                resp = openrouter.call(model, PROMPT, [Path(path)], max_tokens=4000)
                row["response"], row["finish"] = openrouter.text_of(resp)
                row["provider"] = resp.get("provider")
            except RuntimeError as e:
                row["error"] = str(e)[:300]
            rows.append(row)
    return rows

with ThreadPoolExecutor(max_workers=len(MODELS)) as pool:
    naming = pd.DataFrame([row for rows in pool.map(ask_model, MODELS) for row in rows])

naming.to_csv(CHECK / "naming_responses.csv", index=False)
print("calls:", len(naming), "| errors:", naming["error"].notna().sum())
naming.groupby(["model", "version"])["error"].apply(lambda s: s.notna().sum()).unstack()

calls: 400 | errors: 0


version,crop,full
model,,
anthropic/claude-opus-5,0,0
google/gemini-3.1-pro-preview,0,0
meta-llama/llama-4-maverick,0,0
openai/gpt-5.6-sol,0,0
qwen/qwen3-vl-235b-a22b-instruct,0,0


## Draft labels, then check them by hand

The rules only make a first pass:
- **The true surname appears:** `correct`.
- **A refusal phrase:** `refused`.
- **"Don't know":** `unknown`.
- **Anything else:** `check`, which means read it yourself. It is usually a wrong name, sometimes a hedge.

The draft is written only if `naming_labels.csv` doesn't exist yet, so your corrections survive reruns. Open the CSV, replace every `check`, fix any rule mistakes, then run the summary.

In [4]:
import re

SURNAME = {"loof": r"l[öo]{1,2}f", "ohisalo": r"ohisalo"}
REFUSED = r"can['’]?t identify|cannot identify|not able to identify|unable to identify|can['’]?t help identify|don['’]?t identify"
UNKNOWN = r"don['’]?t know|do not know|not sure who|no idea who"

def draft_label(row):
    if isinstance(row["error"], str):
        return "error"
    text = row["response"].lower() if isinstance(row["response"], str) else ""
    if re.search(SURNAME[row["subject"]], text):
        return "correct"
    if re.search(REFUSED, text):
        return "refused"
    if re.search(UNKNOWN, text):
        return "unknown"
    return "check"

LABELS = CHECK / "naming_labels.csv"
if not LABELS.exists():
    draft = naming[["model", "position", "version"]].assign(label=naming.apply(draft_label, axis=1))
    draft.to_csv(LABELS, index=False)
    print("wrote draft labels:", draft["label"].value_counts().to_dict())
else:
    print("naming_labels.csv already exists; not overwritten")

naming_labels.csv already exists; not overwritten


## How the 39 `check` answers were labelled by hand (2 Oct 2026)

- **Names a specific person** (including "likely X"): `wrong_name`. That covers all 30 such answers; none named the right woman.
- **Llama's "The person in the image is unidentifiable / not identifiable":** `refused`.
- **"Not clear enough to identify":** `unknown`.

**`via_text` column:** `True` where a correct name plausibly came from text in the frame rather than the face, meaning either:
- the answer cites a name badge, logo, slogan or podium, or
- the model named nobody correctly on the crops (Claude, GPT, Qwen, Llama).

Photo 38 has a legible name badge, and four models read it. Lööf's "FRAMÅT!" podium photos (24, 28, 40) carry the party name.

## Summary (after the labels are checked)

The key comparison is `wrong_name` and `correct` in the full photos versus the crops. A correct name that disappears on the crop came from text in the frame, not from the face.

In [5]:
labels = pd.read_csv(LABELS)
assert not (labels["label"] == "check").any(), "some labels still say 'check': read those answers and fill them in"
named = naming.merge(labels, on=["model", "position", "version"], validate="one_to_one")

summary = named.pivot_table(index=["model", "version"], columns="label", values="position",
                            aggfunc="count", fill_value=0)
summary

label                                     correct  error  refused  unknown  \
model                            version                                     
anthropic/claude-opus-5          crop           0      0       40        0   
                                 full           1      0       39        0   
google/gemini-3.1-pro-preview    crop          30      1        0        2   
                                 full          35      0        0        2   
meta-llama/llama-4-maverick      crop           0      0        8       24   
                                 full           3      0        8       28   
openai/gpt-5.6-sol               crop           0      0       39        1   
                                 full           1      0       32        7   
qwen/qwen3-vl-235b-a22b-instruct crop           0      0        3       37   
                                 full           3      0        5       21   

label                                     wrong_name  
model                            version              
anthropic/claude-opus-5          crop              0  
                                 full              0  
google/gemini-3.1-pro-preview    crop              7  
                                 full              3  
meta-llama/llama-4-maverick      crop              8  
                                 full              1  
openai/gpt-5.6-sol               crop              0  
                                 full              0  
qwen/qwen3-vl-235b-a22b-instruct crop              0  
                                 full             11

### Correct names from the face alone (crops), by subject

Crops have almost no background, so a correct name here comes from the face. A model that knows a face is a *familiar* viewer in Jenkins' terms.

In [6]:
crop = named[named["version"] == "crop"]
pd.crosstab(crop["model"], [crop["subject"], crop["label"]])

subject                             loof                            ohisalo  \
label                            correct refused unknown wrong_name correct   
model                                                                         
anthropic/claude-opus-5                0      20       0          0       0   
google/gemini-3.1-pro-preview         17       0       0          3      13   
meta-llama/llama-4-maverick            0       4      10          6       0   
openai/gpt-5.6-sol                     0      20       0          0       0   
qwen/qwen3-vl-235b-a22b-instruct       0       2      18          0       0   

subject                                                            
label                            error refused unknown wrong_name  
model                                                              
anthropic/claude-opus-5              0      20       0          0  
google/gemini-3.1-pro-preview        1       0       2          4  
meta-llama/llama-4-maverick          0       4      14          2  
openai/gpt-5.6-sol                   0      19       1          0  
qwen/qwen3-vl-235b-a22b-instruct     0       1      19          0